# 🧬 Modul 01 Hands-on: Network Anatomy & Representations

Selamat datang di Hands-on Modul 01! Pada notebook ini, Anda akan menguji dan membuktikan secara empiris konsep-konsep kunci dari neural network:
1. **The Working Blackbox**: Melihat model neural network memisahkan data non-linear 2D.
2. **Anatomical Dissection**: Memeriksa dimensi tensor (*Shape Contract*), bobot ($W$), dan bias ($b$).
3. **The Sabotage Lab**: Menghapus fungsi aktivasi non-linear dan menyaksikan secara visual bagaimana arsitektur 5-layer runtuh menjadi regresi garis lurus sederhana.
4. **Activation Showdown**: Mengamati karakteristik ReLU, LeakyReLU, Sigmoid, dan GELU.
5. **Hands-on Challenge**: Menyelesaikan tantangan memperbaiki arsitektur jaringan.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import sys
from pathlib import Path

# Hubungkan ke shared visualizer module
sys.path.append(str(Path.cwd().parent.parent))
from shared.visualizer import plot_decision_boundary

# Deteksi Hardware (Otomatis mendeteksi GPU Apple Silicon MPS jika tersedia)
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Menggunakan device akselerasi: {device}")

## 1. Menyiapkan Dataset Non-Linear (Two Moons)
Kita membuat dataset sintetis 2D yang memiliki pola melengkung (*non-linear*). Data ini **mustahil** dipisahkan dengan satu garis lurus.

In [ ]:
# Buat 600 sampel data dua bulan sabit dengan noise
X_np, y_np = make_moons(n_samples=600, noise=0.20, random_state=42)

# Visualisasikan data mentah
plt.figure(figsize=(7, 5))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap=plt.cm.Spectral, edgecolors="k", s=35)
plt.title("Dataset Sintetis: Two Moons (Non-Linearly Separable)", fontsize=12, fontweight="bold")
plt.xlabel("Fitur x1")
plt.ylabel("Fitur x2")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# Konversi ke PyTorch Tensor
X_tensor = torch.tensor(X_np, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1).to(device)  # Shape: [N, 1]
print(f"Bentuk Tensor Input X: {X_tensor.shape} (N_samples, N_features)")
print(f"Bentuk Tensor Label y: {y_tensor.shape} (N_samples, 1)")

## 2. The Working Blackbox: Multi-Layer Perceptron dengan ReLU
Kita bangun jaringan saraf tiruan sederhana:
* Input: 2 fitur ($x_1, x_2$)
* Hidden Layer 1: 16 neuron + aktivasi **ReLU**
* Hidden Layer 2: 16 neuron + aktivasi **ReLU**
* Output Layer: 1 neuron (menghasilkan logit klasifikasi binary)

In [ ]:
class NonLinearMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),   # [Batch, 2] -> [Batch, 16]
            nn.ReLU(),          # Non-linear activation
            nn.Linear(16, 16),  # [Batch, 16] -> [Batch, 16]
            nn.ReLU(),          # Non-linear activation
            nn.Linear(16, 1)    # [Batch, 16] -> [Batch, 1] (Output logit)
        )
        
    def forward(self, x):
        return self.net(x)

# Inisialisasi model, loss function, dan optimizer
model_good = NonLinearMLP().to(device)
criterion = nn.BCEWithLogitsLoss()  # Numerically stable Binary Cross Entropy
optimizer = optim.Adam(model_good.parameters(), lr=0.05)

# Training loop singkat (200 epochs)
print("Training model dengan ReLU...")
for epoch in range(1, 201):
    model_good.train()
    optimizer.zero_grad()
    outputs = model_good(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        preds = (torch.sigmoid(outputs) > 0.5).float()
        acc = (preds == y_tensor).float().mean() * 100
        print(f"Epoch {epoch:3d}/200 | Loss: {loss.item():.4f} | Accuracy: {acc.item():.2f}%")

# Visualisasikan Decision Boundary
plot_decision_boundary(model_good, X_tensor, y_tensor, title="Decision Boundary: MLP dengan ReLU (Berhasil Melengkung!)")

## 3. Anatomical Dissection: Menginspeksi Bobot & Tensor Contract
Mari kita intip parameter dalam network yang baru kita latih:

In [ ]:
print("=== INSPEKSI BOBOT & BIAS LAYER PERTAMA ===")
first_layer = model_good.net[0]  # nn.Linear(2, 16)
print(f"Layer type: {first_layer}")
print(f"Weight matrix shape: {first_layer.weight.shape}  -> [Output_Features, Input_Features]")
print(f"Bias vector shape:   {first_layer.bias.shape}    -> [Output_Features]")

print("\n=== SIMULASI FORWARD PASS STEP-BY-STEP PADA 1 SAMPEL ===")
sample_x = X_tensor[:1]  # Shape: [1, 2]
print(f"Input shape:               {sample_x.shape}")

# Step 1: Linear combination z = xW^T + b
z1 = first_layer(sample_x)
print(f"Setelah Linear(2, 16):      {z1.shape}")

# Step 2: ReLU activation
a1 = model_good.net[1](z1)
print(f"Setelah ReLU:               {a1.shape} (Nilai negatif dipotong jadi 0)")
print(f"Contoh nilai aktivasi a1:\n{a1.detach().cpu().numpy()}")

## 4. The Sabotage Lab: Apa yang Terjadi Jika Kita Menghapus Fungsi Aktivasi?

> **Hipotesis Teori**: *"Tanpa fungsi aktivasi non-linear, berapapun layernya, neural network hanya mampu menghasilkan garis lurus kaku (linear boundary)."*

Mari kita buktikan secara empiris! Kita buat model 5-layer yang **semuanya murni linear tanpa satupun fungsi aktivasi**.

In [ ]:
class PureLinearMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 5 layer linear TANPA aktivasi sama sekali!
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Linear(32, 32),
            nn.Linear(32, 16),
            nn.Linear(16, 8),
            nn.Linear(8, 1)
        )
        
    def forward(self, x):
        return self.net(x)

# Latih model yang dirusak ini dengan dataset yang sama persis
model_sabotaged = PureLinearMLP().to(device)
optimizer_sabotaged = optim.Adam(model_sabotaged.parameters(), lr=0.05)

print("Training model 5-layer TANPA fungsi aktivasi...")
for epoch in range(1, 201):
    model_sabotaged.train()
    optimizer_sabotaged.zero_grad()
    outputs = model_sabotaged(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer_sabotaged.step()
    
    if epoch % 50 == 0:
        preds = (torch.sigmoid(outputs) > 0.5).float()
        acc = (preds == y_tensor).float().mean() * 100
        print(f"Epoch {epoch:3d}/200 | Loss: {loss.item():.4f} | Accuracy: {acc.item():.2f}%")

# Visualisasikan Decision Boundary
plot_decision_boundary(model_sabotaged, X_tensor, y_tensor, title="SABOTAGE LAB: 5-Layer Network TANPA Aktivasi (Garis Kaku!)")

### 💡 Refleksi Sabotage Lab:
Perhatikan gambar di atas!
Meskipun model memiliki **5 layer**, batas pemisahnya tetap **garis lurus kaku**. Akurasi mentok di ~85% dan tidak bisa memisahkan kurva melingkar secara sempurna.

Ini membuktikan secara visual dan matematis: **Fungsi aktivasi adalah satu-satunya komponen yang memberi jaringan saraf kekuatan untuk melengkungkan ruang dan memodelkan dunia nyata yang kompleks.**

## 5. Activation Showdown: ReLU vs Sigmoid vs Tanh vs GELU
Mari kita visualisasikan bagaimana masing-masing fungsi aktivasi merespons rentang nilai dari $-4$ hingga $+4$.

In [ ]:
z = torch.linspace(-4, 4, 200)

activations = {
    "ReLU": nn.ReLU()(z),
    "LeakyReLU (alpha=0.1)": nn.LeakyReLU(0.1)(z),
    "Sigmoid": torch.sigmoid(z),
    "Tanh": torch.tanh(z),
    "GELU (Modern Transformer)": nn.GELU()(z)
}

plt.figure(figsize=(10, 6))
for name, val in activations.items():
    plt.plot(z.numpy(), val.numpy(), label=name, linewidth=2)

plt.title("Perbandingan Karakteristik Fungsi Aktivasi Populer", fontsize=13, fontweight="bold")
plt.xlabel("Input Linear z (Wx + b)")
plt.ylabel("Output Aktivasi a = f(z)")
plt.axhline(0, color="black", linestyle="--", alpha=0.3)
plt.axvline(0, color="black", linestyle="--", alpha=0.3)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=10)
plt.show()

## 6. 🏆 Hands-on Challenge: "The Concentric Circles"

Sekarang giliran Anda! Diberikan dataset berupa dua lingkaran konsentris (lingkaran dalam vs lingkaran luar).

Tugas Anda:
1. Buat class model `CircleClassifier(nn.Module)` menggunakan minimal 2 hidden layer dengan aktivasi **GELU** atau **LeakyReLU**.
2. Latih model hingga akurasi mencapai **minimal 95%**.
3. Tampilkan gambar decision boundary-nya!

In [ ]:
from sklearn.datasets import make_circles

# 1. Dataset Lingkaran Konsentris
X_circ, y_circ = make_circles(n_samples=600, factor=0.4, noise=0.1, random_state=42)
X_circ_tensor = torch.tensor(X_circ, dtype=torch.float32).to(device)
y_circ_tensor = torch.tensor(y_circ, dtype=torch.float32).unsqueeze(1).to(device)

# =========================================================
# TULIS KODE SOLUSI ANDA DI BAWAH INI:
# =========================================================
class CircleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Lengkapi arsitektur layer di sini
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.GELU(),
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.net(x)

# TODO: Inisialisasi model, optimizer, dan jalankan training loop
my_circle_model = CircleClassifier().to(device)
opt = optim.Adam(my_circle_model.parameters(), lr=0.03)

for epoch in range(1, 151):
    my_circle_model.train()
    opt.zero_grad()
    out = my_circle_model(X_circ_tensor)
    loss = criterion(out, y_circ_tensor)
    loss.backward()
    opt.step()

# Evaluasi hasil akurasi dan plot
plot_decision_boundary(my_circle_model, X_circ_tensor, y_circ_tensor, title="Solusi Anda: Circle Classifier")
acc = ((torch.sigmoid(my_circle_model(X_circ_tensor)) > 0.5) == y_circ_tensor).float().mean() * 100
print(f"🎯 Final Accuracy Anda: {acc.item():.2f}%")

## 🏁 Rangkuman Modul 01
* Jaringan saraf adalah fungsi matematika komposisional: $f(x) = f_n(...f_2(f_1(x)))$.
* **Tanpa fungsi aktivasi non-linear, layer yang dalam tidak ada artinya**.
* **ReLU** adalah pilihan default terbaik untuk hidden layer karena cepat dan minim saturasi.
* **GELU** adalah varian modern yang menjadi standar di arsitektur Transformers/LLM.

Setelah berhasil menjalankan notebook ini, jangan lupa tandai kemajuan Anda di [**`TRACKER.md`**](../../TRACKER.md) dan lanjut ke **Modul 02: The Learning Engine & Optimization**!